In [1]:
# 2A

from pathlib import Path
import copy
import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)
torch.set_num_threads(2)
torch.use_deterministic_algorithms(True)

folder = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
data_path = folder / "retail.csv"

if not data_path.is_file() and (folder / "upload" / "retail.csv").is_file():
    data_path = folder / "upload" / "retail.csv"

if not data_path.is_file():
    from tkinter import Tk, filedialog

    root = Tk()
    root.withdraw()
    root.attributes("-topmost", True)
    selected = filedialog.askopenfilename(
        parent=root,
        title="Select retail.csv",
        filetypes=[("CSV files", "*.csv")]
    )
    root.destroy()

    if not selected:
        raise FileNotFoundError("No data file selected.")

    data_path = Path(selected)

columns = ["StockCode", "CustomerID"]
df = pd.read_csv(data_path, dtype={c: "string" for c in columns})

assert not df[columns + ["Quantity"]].isna().any().any()

X = df[columns].astype("category")
y = df["Quantity"].to_numpy(dtype=np.float64)

assert np.isfinite(y).all() and (y > 0).all()

cardinality = X.nunique()
print(cardinality.to_string())
print("One-hot columns:", int(cardinality.sum()))
print(f"Dense float32 size: {len(X) * cardinality.sum() * 4 / 1e9:.2f} GB")

order = np.random.default_rng(42).permutation(len(df))
n_train, n_val = int(0.70 * len(df)), int(0.15 * len(df))
train_idx, val_idx, test_idx = np.split(
    order, [n_train, n_train + n_val]
)

z = np.log1p(y)
z_train = torch.tensor(z[train_idx], dtype=torch.float32)
z_val = z[val_idx]
results = []

print("Train / validation / test:",
      len(train_idx), len(val_idx), len(test_idx))

StockCode     3665
CustomerID    4339
One-hot columns: 8004
Dense float32 size: 12.74 GB
Train / validation / test: 278546 59688 59690


In [2]:
# 2B

def encode_means(fit_idx, apply_idx):
    global_mean = float(y[fit_idx].mean())
    encoded = np.empty((len(apply_idx), 2), dtype=np.float64)

    for j, column in enumerate(columns):
        means = df.iloc[fit_idx].groupby(
            column, observed=True
        )["Quantity"].mean()

        encoded[:, j] = (
            X.iloc[apply_idx][column]
            .map(means)
            .astype(float)
            .fillna(global_mean)
            .to_numpy()
        )

    return encoded


encoded_train = np.empty((len(train_idx), 2), dtype=np.float64)
folds = KFold(n_splits=5, shuffle=True, random_state=42)

for fit_rows, held_rows in folds.split(train_idx):
    encoded_train[held_rows] = encode_means(
        train_idx[fit_rows], train_idx[held_rows]
    )

encoded_val = encode_means(train_idx, val_idx)
encoded_test = encode_means(train_idx, test_idx)

scaler = StandardScaler()

te_train = torch.tensor(
    scaler.fit_transform(np.log1p(encoded_train)),
    dtype=torch.float32
)
te_val = torch.tensor(
    scaler.transform(np.log1p(encoded_val)),
    dtype=torch.float32
)
te_test = torch.tensor(
    scaler.transform(np.log1p(encoded_test)),
    dtype=torch.float32
)


class TargetMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, numerical):
        return self.net(numerical).squeeze(1)


@torch.no_grad()
def predict_log(model, inputs):
    model.eval()
    output = []

    for start in range(0, len(inputs[0]), 16384):
        output.append(
            model(*(x[start:start + 16384] for x in inputs)).numpy()
        )

    prediction = np.concatenate(output).astype(np.float64)
    assert np.isfinite(prediction).all()
    return np.maximum(prediction, 0.0)


def train_model(model, train_inputs, val_inputs):
    nn.init.constant_(
        model.net[-1].bias, float(z[train_idx].mean())
    )
    optimizer = torch.optim.Adam(
        model.parameters(), lr=0.002, weight_decay=1e-5
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=3
    )

    best_loss, stale = float("inf"), 0

    for epoch in range(1, 81):
        model.train()

        for batch in torch.randperm(len(z_train)).split(2048):
            prediction = model(*(x[batch] for x in train_inputs))
            loss = nn.functional.mse_loss(
                prediction, z_train[batch]
            )

            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite training loss.")

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()

        val_loss = float(np.mean(
            (z_val - predict_log(model, val_inputs)) ** 2
        ))
        scheduler.step(val_loss)

        if val_loss < best_loss:
            best_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            stale = 0
        else:
            stale += 1

        if stale >= 8:
            break

    model.load_state_dict(best_state)
    print(
        f"Selected epoch: {best_epoch}; "
        f"validation RMSLE: {np.sqrt(best_loss):.6f}"
    )
    return model


def evaluate(name, prediction_log):
    prediction = np.expm1(prediction_log)

    row = {
        "Model": name,
        "Test RMSLE": float(np.sqrt(np.mean(
            (z[test_idx] - prediction_log) ** 2
        ))),
        "Test MAE": float(np.mean(
            np.abs(y[test_idx] - prediction)
        ))
    }

    results.append(row)
    print(pd.DataFrame([row]).to_string(
        index=False, float_format=lambda v: f"{v:.6f}"
    ))


torch.manual_seed(42)
target_model = train_model(
    TargetMLP(), (te_train,), (te_val,)
)
target_log = predict_log(target_model, (te_test,))
evaluate("Target encoding", target_log)

Selected epoch: 41; validation RMSLE: 0.621434
          Model  Test RMSLE  Test MAE
Target encoding    0.619597  7.043412


In [3]:
# 2C

category_ids = np.zeros((len(X), 2), dtype=np.int64)
vocab_sizes = []

for j, column in enumerate(columns):
    categories = X.iloc[train_idx][column].unique()
    mapping = {
        value: i + 1 for i, value in enumerate(categories)
    }

    category_ids[:, j] = (
        X[column]
        .map(mapping)
        .astype(float)
        .fillna(0)
        .to_numpy(dtype=np.int64)
    )

    vocab_sizes.append(len(categories) + 1)

ids_train = torch.from_numpy(category_ids[train_idx])
ids_val = torch.from_numpy(category_ids[val_idx])
ids_test = torch.from_numpy(category_ids[test_idx])


class EmbeddingMLP(nn.Module):
    def __init__(self):
        super().__init__()

        self.embeddings = nn.ModuleList([
            nn.Embedding(size, 16, padding_idx=0)
            for size in vocab_sizes
        ])

        self.net = nn.Sequential(
            nn.Linear(32, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, ids):
        vectors = torch.cat([
            embedding(ids[:, j])
            for j, embedding in enumerate(self.embeddings)
        ], dim=1)

        return self.net(vectors).squeeze(1)


torch.manual_seed(42)
embedding_model = train_model(
    EmbeddingMLP(), (ids_train,), (ids_val,)
)
embedding_log = predict_log(embedding_model, (ids_test,))
evaluate("Embeddings", embedding_log)

Selected epoch: 33; validation RMSLE: 0.532317
     Model  Test RMSLE  Test MAE
Embeddings    0.527305  6.105479


In [5]:
# 2D

class CombinedMLP(EmbeddingMLP):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(34, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, ids, numerical):
        vectors = torch.cat([
            embedding(ids[:, j])
            for j, embedding in enumerate(self.embeddings)
        ], dim=1)

        combined = torch.cat([vectors, numerical], dim=1)
        return self.net(combined).squeeze(1)


torch.manual_seed(42)
combined_model = train_model(
    CombinedMLP(),
    (ids_train, te_train),
    (ids_val, te_val)
)

combined_log = predict_log(
    combined_model, (ids_test, te_test)
)
evaluate("Combined", combined_log)

Selected epoch: 17; validation RMSLE: 0.527261
   Model  Test RMSLE  Test MAE
Combined    0.524706  6.021976
